# 99 Build the Part 3 planner on the real data

The Part 3 demand-flexibility and battery planner (`web/part3/`) is a single web page that opens straight in a
browser, with no server. It doesn't read the database while it runs: its data are built into the page. This notebook
does that build, on the results of the earlier notebooks:

1. **Export** (`scripts/export_part3_inputs.py`): the hourly loads and one row per customer (segment, programme
   flags, appliance types and confidences, technology scores, list flags, mid-year change flags) from `avangrid.db`
   and the cached arrays, to `onedrive-export/Data/part3/`.
2. **Build, once per utility** (`scripts/run_part3.py --utility ...`): the planner's calculations (system peak,
   flexible loads, battery sizing, customer economics, ranking) with the assumptions in
   `config/part3_assumptions.json`, but priced on that utility's real residential time-varying rate from
   `data/tariffs/residential_tariffs.json` (scripts/fetch_tariffs.py). Each utility gets its own page.
3. **A starting page** with a drop-down to switch between the four utilities' planners.

Prices change bill savings, payback and NPV, and through them each home's economics score, rank and programme
channel, so every utility needs its own build; battery sizing and the peak reduction don't depend on prices and are
the same in all four. The meter data come from one service area (the challenge documents don't say which), so the
four pages answer "what if these homes were on this utility's rate".

At the end, open **`onedrive-export/Data/part3/index.html`** in a browser (the last cell prints a link). Each page
lists the top 3,000 customers individually (set `MAX_CUSTOMERS` in the setup cell to change that); its summary
figures cover everyone.

**The page holds real customer data.** It is written inside the git-ignored data folder; don't commit it or share it
outside the event. The committed `web/part3/index.html` uses a synthetic demonstration dataset and is safe to share.

Run after notebooks 10–70 (the export needs `technology_predictions`, `appliance_inventory`, `customer_segments` and
`change_points`). How each step works is described in `web/part3/README.md` and on the page's Methodology tab.

## Setup

In [1]:
import json
import subprocess
import sys
import time
from pathlib import Path

import polars as pl
from IPython.display import Markdown, display

from avangrid import config

ROOT = Path.cwd()
PART3 = config.DATA_DIR / 'part3'
# Utilities to build, in drop-down order, and the rate each one uses (see data/tariffs/README.md).
UTILITIES = {'cmp': 'CMP: Time-of-use (A-TOU)', 'nyseg': 'NYSEG (West): Day/Night (SC 8)',
             'rge': 'RG&E: Time-of-use (SC 4)', 'ui': 'UI: Time-of-use (RT)'}
# Customers listed individually on the page (the top-ranked ones). Every summary figure still covers all customers.
# 3,000 gives a page of about 30 MB that opens quickly; 0 lists everyone (about 160 MB, slow to open).
MAX_CUSTOMERS = 3000


def run(script, *args):
    """Run a project script with this notebook's Python, show its output, and stop the notebook if it fails."""
    t = time.time()
    r = subprocess.run([sys.executable, str(ROOT / 'scripts' / script), *map(str, args)], cwd=ROOT,
                       capture_output=True, text=True)
    print(r.stdout[-4000:] + (('\n' + r.stderr[-4000:]) if r.returncode else ''))
    if r.returncode:
        raise RuntimeError(f'{script} failed (exit code {r.returncode})')
    print(f'{script}: done in {time.time() - t:.0f}s')

## 1. Export the inputs from the database

In [2]:
run('export_part3_inputs.py')
manifest = json.loads((PART3 / 'manifest.json').read_text())
{k: v for k, v in manifest.items() if not isinstance(v, (dict, list))}

Loading hourly arrays (cached)...
Filling gaps for 33,310 customers x 365 days...
Joining Part 1 and Part 2 results...
Wrote 33,310 customers x 8,760 hours to /Users/nchaturvedi/Documents/Code/avangrid/onedrive-export/Data/part3
  hourly_kwh.npy 1.17 GB, customers.csv 61 columns

export_part3_inputs.py: done in 7s


{'generated': '2026-11-13T23:41:26',
 'source': 'avangrid-hackathon repo (notebooks 10-70)',
 'n_customers': 33310,
 'n_customers_available': 33310,
 'sample': False,
 'n_hours': 8760,
 'first_date': '2025-09-01',
 'last_date': '2026-08-31'}

## 2. Build the planner for each utility

Each build runs the whole Part 3 method on every customer (about a minute each) and writes three files to
`onedrive-export/Data/part3/<utility>/`: `app_data.json` (everything the page reads), `index.html` (the page, with
that data built in) and `candidates.csv` (one row per customer with the main results, ranked).

How each utility's rate is converted to the planner's format is described in `utility_rates` in
`scripts/run_part3.py`: the page models a rate as on- and off-peak prices per season plus one flat supply price, and
the four time-varying rates used here fit that exactly.

In [3]:
for utility in UTILITIES:
    run('run_part3.py', '--utility', utility, '--output', PART3 / utility,
        *(['--max-customers', MAX_CUSTOMERS] if MAX_CUSTOMERS else []))
for old in ('app_data.json', 'candidates.csv'):          # outputs of an earlier single build, now per utility
    (PART3 / old).unlink(missing_ok=True)

rates: CMP Time-of-use (Rate A-TOU), effective 2026-01-01 (OpenEI; delivery and supply combined)
33,310 customers x 8,760 hours
  windows: summer HE [17, 18, 19, 20], winter HE [18, 19, 20, 21]; binding season summer  (1s)
  1,000/33,310 customers  (2s)
  2,000/33,310 customers  (3s)
  3,000/33,310 customers  (4s)
  4,000/33,310 customers  (5s)
  5,000/33,310 customers  (7s)
  6,000/33,310 customers  (8s)
  7,000/33,310 customers  (9s)
  8,000/33,310 customers  (10s)
  9,000/33,310 customers  (11s)
  10,000/33,310 customers  (13s)
  11,000/33,310 customers  (14s)
  12,000/33,310 customers  (15s)
  13,000/33,310 customers  (16s)
  14,000/33,310 customers  (17s)
  15,000/33,310 customers  (19s)
  16,000/33,310 customers  (20s)
  17,000/33,310 customers  (21s)
  18,000/33,310 customers  (22s)
  19,000/33,310 customers  (23s)
  20,000/33,310 customers  (25s)
  21,000/33,310 customers  (26s)
  22,000/33,310 customers  (27s)
  23,000/33,310 customers  (28s)
  24,000/33,310 customers  (29s)
 

rates: NYSEG Day/Night (SC 8) (West), effective 2025-02-26 (OpenEI; delivery and supply combined)
33,310 customers x 8,760 hours
  windows: summer HE [17, 18, 19, 20], winter HE [18, 19, 20, 21]; binding season summer  (1s)
  1,000/33,310 customers  (2s)
  2,000/33,310 customers  (3s)
  3,000/33,310 customers  (4s)
  4,000/33,310 customers  (6s)
  5,000/33,310 customers  (7s)
  6,000/33,310 customers  (8s)
  7,000/33,310 customers  (9s)
  8,000/33,310 customers  (10s)
  9,000/33,310 customers  (12s)
  10,000/33,310 customers  (13s)
  11,000/33,310 customers  (14s)
  12,000/33,310 customers  (15s)
  13,000/33,310 customers  (17s)
  14,000/33,310 customers  (18s)
  15,000/33,310 customers  (19s)
  16,000/33,310 customers  (20s)
  17,000/33,310 customers  (22s)
  18,000/33,310 customers  (23s)
  19,000/33,310 customers  (24s)
  20,000/33,310 customers  (25s)
  21,000/33,310 customers  (27s)
  22,000/33,310 customers  (28s)
  23,000/33,310 customers  (29s)
  24,000/33,310 customers  (30s)


rates: RG&E Time-of-use (SC 4, Schedule I), effective 2025-05-01 (OpenEI; delivery and supply combined)
33,310 customers x 8,760 hours
  windows: summer HE [17, 18, 19, 20], winter HE [18, 19, 20, 21]; binding season summer  (1s)
  1,000/33,310 customers  (2s)
  2,000/33,310 customers  (3s)
  3,000/33,310 customers  (4s)
  4,000/33,310 customers  (5s)
  5,000/33,310 customers  (7s)
  6,000/33,310 customers  (8s)
  7,000/33,310 customers  (9s)
  8,000/33,310 customers  (10s)
  9,000/33,310 customers  (11s)
  10,000/33,310 customers  (13s)
  11,000/33,310 customers  (14s)
  12,000/33,310 customers  (15s)
  13,000/33,310 customers  (16s)
  14,000/33,310 customers  (17s)
  15,000/33,310 customers  (19s)
  16,000/33,310 customers  (20s)
  17,000/33,310 customers  (21s)
  18,000/33,310 customers  (22s)
  19,000/33,310 customers  (23s)
  20,000/33,310 customers  (25s)
  21,000/33,310 customers  (26s)
  22,000/33,310 customers  (27s)
  23,000/33,310 customers  (28s)
  24,000/33,310 customers  

rates: UI Time-of-use (Rate RT), effective 2026-01-01 (OpenEI; delivery and supply combined)
33,310 customers x 8,760 hours
  windows: summer HE [17, 18, 19, 20], winter HE [18, 19, 20, 21]; binding season summer  (1s)
  1,000/33,310 customers  (2s)
  2,000/33,310 customers  (3s)
  3,000/33,310 customers  (4s)
  4,000/33,310 customers  (6s)
  5,000/33,310 customers  (7s)
  6,000/33,310 customers  (8s)
  7,000/33,310 customers  (10s)
  8,000/33,310 customers  (11s)
  9,000/33,310 customers  (12s)
  10,000/33,310 customers  (13s)
  11,000/33,310 customers  (15s)
  12,000/33,310 customers  (16s)
  13,000/33,310 customers  (17s)
  14,000/33,310 customers  (18s)
  15,000/33,310 customers  (20s)
  16,000/33,310 customers  (21s)
  17,000/33,310 customers  (22s)
  18,000/33,310 customers  (23s)
  19,000/33,310 customers  (25s)
  20,000/33,310 customers  (26s)
  21,000/33,310 customers  (27s)
  22,000/33,310 customers  (28s)
  23,000/33,310 customers  (30s)
  24,000/33,310 customers  (31s)
  25

### The four utilities compared

The same homes, batteries and peak reduction, priced on each utility's rate. Battery savings are year-1 bill
savings from the battery alone (after flexible loads), for homes recommended a battery; the break-even rebate is the
upfront payment that would make the battery worth it to the customer (NPV = 0).

In [4]:
rows = []
for utility, label in UTILITIES.items():
    meta = json.loads((PART3 / utility / 'app_data.json').read_text())['config']['economics']['tariff']
    c = pl.read_csv(PART3 / utility / 'candidates.csv')
    b = c.filter(pl.col('units') > 0)
    rows.append({'utility': label, 'on_peak_hours': ', '.join(map(str, json.loads(
                     (PART3 / utility / 'app_data.json').read_text())['config']['economics']['rates']['summer']['on_peak_he'])),
                 'battery_homes': b.height, 'median_battery_savings': b['batt_savings'].median(),
                 'homes_irr_above_7pct': int((b['irr'] > 0.07).sum()),
                 'median_breakeven_rebate': b['breakeven_rebate'].median(),
                 'wave_1_homes': c.filter(pl.col('wave') == 'Wave 1').height, 'rate_effective': meta['effective']})
compare = pl.DataFrame(rows)
with pl.Config(tbl_width_chars=200, tbl_hide_dataframe_shape=True, float_precision=0, fmt_str_lengths=60):
    print(compare)

┌────────────────────────────────┬───────────────────────────────────────────┬───────────────┬────────────────────────┬──────────────────────┬─────────────────────────┬──────────────┬────────────────┐
│ utility                        ┆ on_peak_hours                             ┆ battery_homes ┆ median_battery_savings ┆ homes_irr_above_7pct ┆ median_breakeven_rebate ┆ wave_1_homes ┆ rate_effective │
│ ---                            ┆ ---                                       ┆ ---           ┆ ---                    ┆ ---                  ┆ ---                     ┆ ---          ┆ ---            │
│ str                            ┆ str                                       ┆ i64           ┆ f64                    ┆ i64                  ┆ f64                     ┆ i64          ┆ str            │
╞════════════════════════════════╪═══════════════════════════════════════════╪═══════════════╪════════════════════════╪══════════════════════╪═════════════════════════╪══════════════╪═════════════

## 3. The starting page, with a utility drop-down

`onedrive-export/Data/part3/index.html` shows one utility's planner at a time and switches with the drop-down
(the choice is kept in the address, e.g. `index.html#ui`). Each utility's page can also be opened on its own.
Everything is local files: no internet connection or server is needed.

In [5]:
options = '\n'.join(f'<option value="{u}">{label}</option>' for u, label in UTILITIES.items())
chooser = f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Part 3 planner by utility</title>
<style>
  :root {{ color-scheme: light dark; }}
  body {{ margin: 0; font: 14px/1.4 system-ui, sans-serif; display: flex; flex-direction: column; height: 100vh; }}
  header {{ display: flex; gap: 12px; align-items: center; flex-wrap: wrap; padding: 8px 16px;
            border-bottom: 1px solid #8884; }}
  select {{ font: inherit; padding: 4px 8px; }}
  .note {{ color: #888; font-size: 12px; }}
  iframe {{ flex: 1; border: 0; width: 100%; }}
</style></head>
<body>
<header>
  <label for="u"><b>Utility rate</b></label>
  <select id="u">{options}</select>
  <a id="open" target="_blank">Open on its own</a>
  <span class="note">Same homes and batteries; savings, payback and ranking priced on each utility's residential
  time-varying rate. Real customer data: do not share outside the event.</span>
</header>
<iframe id="f" title="Part 3 planner"></iframe>
<script>
  const sel = document.getElementById('u'), frame = document.getElementById('f'), open = document.getElementById('open');
  function show(u) {{ sel.value = u; frame.src = u + '/index.html'; open.href = u + '/index.html'; location.hash = u; }}
  sel.addEventListener('change', () => show(sel.value));
  const start = location.hash.slice(1);
  show([...sel.options].some(o => o.value === start) ? start : sel.options[0].value);
</script>
</body></html>
"""
(PART3 / 'index.html').write_text(chooser)
for utility in UTILITIES:
    page = PART3 / utility / 'index.html'
    print(f'{utility}: {page.stat().st_size / 1e6:.1f} MB')
top = pl.read_csv(PART3 / 'cmp' / 'candidates.csv')
print(f'\nTop of the ranking on CMP\'s rate ({top.height:,} customers ranked):')
with pl.Config(tbl_cols=12, tbl_width_chars=200, tbl_hide_dataframe_shape=True):
    print(top.head(10))
page = PART3 / 'index.html'
display(Markdown(f'**Open the planner:** [{page}]({page.resolve().as_uri()})'))

cmp: 28.8 MB
nyseg: 28.8 MB
rge: 28.8 MB
ui: 28.8 MB

Top of the ranking on CMP's rate (33,310 customers ranked):
┌──────┬──────────┬─────────────────────────────────┬────────────────┬───────┬────────┬───┬─────────────┬───────────────┬──────┬────────┬─────────┬─────────────────────────────────┐
│ rank ┆ customer ┆ segment                         ┆ action         ┆ units ┆ wave   ┆ … ┆ p_heat_pump ┆ p_resist_heat ┆ p_ac ┆ p_pool ┆ p_solar ┆ reason                          │
│ ---  ┆ ---      ┆ ---                             ┆ ---            ┆ ---   ┆ ---    ┆   ┆ ---         ┆ ---           ┆ ---  ┆ ---    ┆ ---     ┆ ---                             │
│ i64  ┆ i64      ┆ str                             ┆ str            ┆ i64   ┆ str    ┆   ┆ f64         ┆ f64           ┆ f64  ┆ f64    ┆ f64     ┆ str                             │
╞══════╪══════════╪═════════════════════════════════╪════════════════╪═══════╪════════╪═══╪═════════════╪═══════════════╪══════╪════════╪═════════╪═══════════

**Open the planner:** [/Users/nchaturvedi/Documents/Code/avangrid/onedrive-export/Data/part3/index.html](file:///Users/nchaturvedi/Documents/Code/avangrid/onedrive-export/Data/part3/index.html)

## Tried but not used

| Tried | Why it isn't used |
|---|---|
| **A live web app reading the database** (the earlier battery simulator: a FastAPI server, `app/server.py`, and its page, `web/index.html`, plus a Streamlit prototype) | Removed in favour of the Part 3 planner. A page with its data built in needs no server, port or running database, so judges only need a browser, and it can be opened on a machine without the data. The trade-off: changing an assumption in `config/part3_assumptions.json` means re-running this notebook. |